# Parkinson's Disease Classifier
Enhanced ensemble model for detecting Parkinson's disease from spiral and wave drawings

In [1]:
# Test imports and install if needed
import sys
import subprocess

def test_import(module_name, install_name=None):
    if install_name is None:
        install_name = module_name
    try:
        __import__(module_name)
        print(f"✓ {module_name} is available")
        return True
    except ImportError:
        print(f"✗ {module_name} not found, installing {install_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", install_name])
        try:
            __import__(module_name)
            print(f"✓ {module_name} installed successfully")
            return True
        except ImportError:
            print(f"✗ Failed to import {module_name} after installation")
            return False

# Test all required modules
modules = [
    ('h5py', 'h5py'),
    ('cv2', 'opencv-python'),
    ('sklearn', 'scikit-learn'),
    ('xgboost', 'xgboost'),
    ('skimage', 'scikit-image')
]

all_good = True
for module, install in modules:
    if not test_import(module, install):
        all_good = False

if all_good:
    print("\n🎉 All packages are ready!")
else:
    print("\n⚠️ Some packages failed to install. Please restart the kernel and try again.")

✗ h5py not found, installing h5py...
✓ h5py installed successfully
✓ cv2 is available
✓ sklearn is available
✓ xgboost is available
✓ skimage is available

🎉 All packages are ready!


In [2]:
# Import all required libraries
import os
import cv2
import numpy as np
import h5py
import pickle
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from skimage.feature import graycomatrix, graycoprops
import warnings
warnings.filterwarnings('ignore')

# Configuration
TARGET_SIZE = (128, 128)
MODEL_H5_PATH = 'parkinson_model.h5'
DATA_DIR = 'Parkinson data'

print("✓ All imports successful!")
print(f"✓ OpenCV version: {cv2.__version__}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ H5PY version: {h5py.__version__}")

✓ All imports successful!
✓ OpenCV version: 4.12.0
✓ NumPy version: 2.4.1
✓ H5PY version: 3.15.1


In [3]:
def preprocess_image(path):
    """Load and preprocess image with CLAHE enhancement"""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: 
        return None
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return cv2.resize(clahe.apply(img), TARGET_SIZE)

def extract_features(img):
    """Extract comprehensive feature set from image"""
    # Basic statistical features
    features = [
        np.mean(img), np.std(img), np.median(img), np.min(img), np.max(img),
        np.percentile(img, 10), np.percentile(img, 25), 
        np.percentile(img, 75), np.percentile(img, 90), np.var(img)
    ]
    
    # Histogram features
    hist, _ = np.histogram(img.flatten(), bins=16, range=(0, 256))
    features.extend((hist / (hist.sum() + 1e-7)).tolist())
    
    # Texture features (GLCM)
    try:
        glcm = graycomatrix(img, [1], [0], 256, True, True)
        for prop in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']:
            features.append(graycoprops(glcm, prop)[0, 0])
    except:
        features.extend([0] * 6)
    
    # Edge features
    edges = cv2.Canny(img, 50, 150)
    features.extend([
        np.mean(edges), np.std(edges), 
        np.sum(edges > 0) / edges.size, np.max(edges)
    ])
    
    # Gradient features
    sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    features.extend([
        np.mean(np.abs(sobel_x)), np.mean(np.abs(sobel_y)),
        np.std(sobel_x), np.std(sobel_y)
    ])
    
    return features

print("✓ Preprocessing functions defined!")

✓ Preprocessing functions defined!


In [4]:
def load_data(base_dir=DATA_DIR):
    """Load all images and labels from dataset"""
    images, labels = [], []
    
    for split in ['training', 'testing']:
        for label, value in [('parkinson', 1), ('healthy', 0)]:
            pattern = os.path.join(base_dir, '**', split, label, '*.*')
            for file_path in glob(pattern, recursive=True):
                img = preprocess_image(file_path)
                if img is not None:
                    images.append(img)
                    labels.append(value)
    
    print(f'Loaded: {sum(labels)} Parkinson, {len(labels) - sum(labels)} Healthy')
    return images, labels

def augment_data(images, labels):
    """Apply data augmentation techniques"""
    aug_images, aug_labels = list(images), list(labels)
    
    for img, label in zip(images, labels):
        # Horizontal flip
        aug_images.append(cv2.flip(img, 1))
        aug_labels.append(label)
        
        # Rotations
        for angle in [5, -5]:
            M = cv2.getRotationMatrix2D((64, 64), angle, 1.0)
            rotated = cv2.warpAffine(img, M, TARGET_SIZE)
            aug_images.append(rotated)
            aug_labels.append(label)
    
    print(f'Augmented: {len(images)} -> {len(aug_images)} samples')
    return aug_images, aug_labels

print("✓ Data loading functions defined!")

✓ Data loading functions defined!


In [5]:
# Load and process data
print("Loading data...")
images, labels = load_data()

print("Augmenting data...")
images, labels = augment_data(images, labels)

print("Extracting features...")
X = np.array([extract_features(img) for img in images])
y = np.array(labels)
print(f'Feature matrix shape: {X.shape}')

Loading data...
Loaded: 204 Parkinson, 204 Healthy
Augmenting data...
Augmented: 408 -> 1632 samples
Extracting features...
Feature matrix shape: (1632, 40)


In [6]:
# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')

Train: 1305 | Test: 327


In [7]:
# Train ensemble model
print("Training ensemble model...")

rf = RandomForestClassifier(
    n_estimators=300, max_depth=20, min_samples_split=3, 
    random_state=42, n_jobs=-1
)
xgb = XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1, 
    subsample=0.8, random_state=42, n_jobs=-1
)
gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1, 
    random_state=42
)

model = VotingClassifier(
    estimators=[('rf', rf), ('xgb', xgb), ('gb', gb)], 
    voting='soft', n_jobs=-1
)

model.fit(X_train_scaled, y_train)
print("✓ Training completed!")

Training ensemble model...
✓ Training completed!


In [8]:
# Evaluate model
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f'\n🎯 Model Accuracy: {accuracy:.2%}')
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Healthy', 'Parkinson']))


🎯 Model Accuracy: 98.78%

📊 Classification Report:
              precision    recall  f1-score   support

     Healthy       1.00      0.98      0.99       164
   Parkinson       0.98      1.00      0.99       163

    accuracy                           0.99       327
   macro avg       0.99      0.99      0.99       327
weighted avg       0.99      0.99      0.99       327



In [9]:
# Save model
print(f"💾 Saving model to {MODEL_H5_PATH}...")

with h5py.File(MODEL_H5_PATH, 'w') as h5f:
    h5f.attrs['model_name'] = 'Enhanced_Ensemble'
    h5f.attrs['accuracy'] = float(accuracy)
    
    # Save scaler
    scaler_group = h5f.create_group('scaler')
    scaler_group.create_dataset('mean', data=scaler.mean_)
    scaler_group.create_dataset('scale', data=scaler.scale_)
    
    # Save model
    pickle_group = h5f.create_group('pickle_model')
    model_data = pickle.dumps({'model': model, 'scaler': scaler})
    pickle_group.create_dataset('data', data=np.frombuffer(model_data, dtype=np.uint8))

print(f"✅ Model saved successfully!")

💾 Saving model to parkinson_model.h5...
✅ Model saved successfully!


In [10]:
# Test loading saved model
def load_h5_model(path):
    """Load model from HDF5 file"""
    with h5py.File(path, 'r') as h:
        return pickle.loads(bytes(h['pickle_model']['data'][:]))

# Load and test
print("🔄 Testing saved model...")
data = load_h5_model(MODEL_H5_PATH)
loaded_model, loaded_scaler = data['model'], data['scaler']

# Verify accuracy
loaded_accuracy = accuracy_score(y_test, loaded_model.predict(X_test_scaled))
print(f'✅ Loaded model accuracy: {loaded_accuracy:.2%}')

🔄 Testing saved model...
✅ Loaded model accuracy: 98.78%


In [11]:
# Sample predictions
def predict_image(path, model, scaler):
    """Predict single image"""
    img = preprocess_image(path)
    if img is None:
        return None
    
    features = np.array(extract_features(img)).reshape(1, -1)
    prediction = model.predict(scaler.transform(features))[0]
    probabilities = model.predict_proba(scaler.transform(features))[0]
    
    return {
        'prediction': 'Parkinson' if prediction == 1 else 'Healthy',
        'confidence': probabilities[prediction]
    }

print("🔍 Testing sample predictions:")
sample_paths = []
for label in ['parkinson', 'healthy']:
    pattern = os.path.join(DATA_DIR, '**', 'testing', label, '*.*')
    sample_paths.extend(glob(pattern, recursive=True)[:2])

for path in sample_paths:
    result = predict_image(path, loaded_model, loaded_scaler)
    if result:
        filename = os.path.basename(path)[:25]
        pred = result['prediction']
        conf = result['confidence']
        print(f"  📄 {filename} -> {pred} ({conf:.1%})")

🔍 Testing sample predictions:
  📄 V01PE01.png -> Parkinson (96.2%)
  📄 V02PE01.png -> Parkinson (97.7%)
  📄 V01HE01.png -> Healthy (98.1%)
  📄 V02HE01.png -> Healthy (97.8%)


## 🎉 Summary

The enhanced Parkinson's disease classifier has been successfully trained with:

- **🎯 High accuracy** (typically >95%)
- **🤖 Ensemble approach** combining Random Forest, XGBoost, and Gradient Boosting
- **📊 Comprehensive features** including statistical, texture, edge, and gradient features
- **🔄 Data augmentation** to improve model robustness
- **💾 Model persistence** saved in HDF5 format for future use

The model can now classify spiral and wave drawings to detect Parkinson's disease with high confidence.